In [18]:
from presidio_analyzer import AnalyzerEngine, PatternRecognizer, Pattern
from presidio_analyzer.nlp_engine import NlpEngineProvider
from presidio_anonymizer import AnonymizerEngine
import re

In [2]:
# Model Provider
engine_name = 'spacy'
model = 'en_core_web_sm'
configuration = {
    "nlp_engine_name": engine_name,
    "models": [{"lang_code": "en", "model_name": model}],}

In [3]:
provider = NlpEngineProvider(nlp_configuration=configuration)
nlp_engine = provider.create_engine()

In [4]:
analyzer = AnalyzerEngine(nlp_engine=nlp_engine)
anonimizer = AnonymizerEngine()


In [ ]:
"""
Detects PII in the given text and replaces it with redacted tags.

By default, the Microsoft Presidio English analyzer detects and redacts the following
- CREDIT_CARD
- CRYPTO
- DATE_TIME
- EMAIL_ADDRESS
- IBAN_CODE
- IP_ADDRESS
- NRPI (Nationality, religious or political group)
- LOCATION
- PERSON
- PHONE_NUMBER
- MEDICAL_LICENSE
- URL
- US_BANK_NUMBER
- US_DRIVER_LICENSE
- US_ITIN
- US_PASSPORT
- US_SSN
"""


In [7]:
results = analyzer.analyze(text="My name is John Doe and my email is john.doe@example.com", language = 'en')
print(results)

[type: EMAIL_ADDRESS, start: 36, end: 56, score: 1.0, type: PERSON, start: 11, end: 19, score: 0.85, type: URL, start: 36, end: 43, score: 0.5, type: URL, start: 45, end: 56, score: 0.5]


In [10]:
redacted_text = anonimizer.anonymize(text="My name is John Doe and my email is john.doe@example.com", analyzer_results = results)

print(redacted_text)

text: My name is <PERSON> and my email is <EMAIL_ADDRESS>
items:
[
    {'start': 36, 'end': 51, 'entity_type': 'EMAIL_ADDRESS', 'text': '<EMAIL_ADDRESS>', 'operator': 'replace'},
    {'start': 11, 'end': 19, 'entity_type': 'PERSON', 'text': '<PERSON>', 'operator': 'replace'}
]



In [15]:
texts = [
    "The sales in the last quarter were $1,000,000. The next quarter is expected to be $1,500,000. The increase is due to the new product launch.",
    "My name is John Doe and my email is john.doe@example.com",
    "My phone number is 123-456-7890",
    "AIRAG_Reference: 1234567890",
    "My persona url is https://www.example.com",
    "I am the admin for the server auasdasd@hostname.com with IP address 123.456.789.0",
]

In [ ]:
for text in texts:
    results = analyzer.analyze(text=text, language = 'en')
    redacted_text = anonimizer.anonymize(text=text, analyzer_results = results)
    print(f"Original Text: {text}")
    
    
    print(f"Redacted Text: {redacted_text}")
    #print(re.sub(r'<[^>]*>', '', redacted_text))
    print("--------------------------------------------------") 

Original Text: The sales in the last quarter were $1,000,000. The next quarter is expected to be $1,500,000. The increase is due to the new product launch.
Redacted Text: text: The sales in <DATE_TIME> were $1,000,000. <DATE_TIME> is expected to be $1,500,000. The increase is due to the new product launch.
items:
[
    {'start': 42, 'end': 53, 'entity_type': 'DATE_TIME', 'text': '<DATE_TIME>', 'operator': 'replace'},
    {'start': 13, 'end': 24, 'entity_type': 'DATE_TIME', 'text': '<DATE_TIME>', 'operator': 'replace'}
]



TypeError: expected string or bytes-like object, got 'EngineResult'